# Demo 04: Linux CUDA LoRA fine-tuning for parametric memory

This notebook fine-tunes local Qwen3-1.7B on factual questions about Luigi Saetta. It is intended for one CUDA-capable NVIDIA GPU with BF16 support.

Run from a clean kernel. The notebook reads and writes only local, ignored artifacts.

In [ ]:
from collections import Counter
import json
from pathlib import Path
import re

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

In [ ]:
def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing AGENTS.md and requirements.txt."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


def require_cuda_bfloat16_support() -> torch.device:
    """Return the CUDA device or raise when CUDA BF16 training is unavailable."""
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Install a CUDA-enabled PyTorch build and run on a supported NVIDIA GPU.')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('This CUDA GPU does not support BF16 training. Use a BF16-capable NVIDIA GPU.')
    return torch.device('cuda')


def to_prompt_completion(record: dict) -> dict:
    """Convert one conversational record to TRL prompt/completion format."""
    return {
        'prompt': record['messages'][:-1],
        'completion': [record['messages'][-1]],
    }

## Editable local artifact directories

Set the three directories in the next cell for the Linux host before running the notebook. The project-relative defaults use ignored `artifacts/` paths.

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# EDIT THESE PATHS for the Linux host. DATASET_DIRECTORY must contain train.jsonl and eval.jsonl.
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-1.7B'
DATASET_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'datasets' / 'cv-qa'
TRAINING_OUTPUT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'training' / 'demo04-qwen3-1.7b-lora-cuda'

TRAIN_FILE = DATASET_DIRECTORY / 'train.jsonl'
EVAL_FILE = DATASET_DIRECTORY / 'eval.jsonl'
ADAPTER_OUTPUT_DIRECTORY = TRAINING_OUTPUT_DIRECTORY / 'adapter'

for required_path in (MODEL_DIRECTORY / 'config.json', TRAIN_FILE, EVAL_FILE):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required_path}')

single_weight_file = MODEL_DIRECTORY / 'model.safetensors'
weight_index_file = MODEL_DIRECTORY / 'model.safetensors.index.json'
if not single_weight_file.is_file() and not weight_index_file.is_file():
    raise FileNotFoundError(
        f'Missing model weights in {MODEL_DIRECTORY}. Expected model.safetensors or model.safetensors.index.json.'
    )
if weight_index_file.is_file():
    shard_names = set(json.loads(weight_index_file.read_text(encoding='utf-8'))['weight_map'].values())
    missing_shards = sorted(name for name in shard_names if not (MODEL_DIRECTORY / name).is_file())
    if missing_shards:
        raise FileNotFoundError(f'Missing model weight shards in {MODEL_DIRECTORY}: {missing_shards}')

print(f'Model directory: {MODEL_DIRECTORY}')
print(f'Dataset directory: {DATASET_DIRECTORY}')
print(f'Training output directory: {TRAINING_OUTPUT_DIRECTORY}')

## Editable training configuration

The base model and LoRA adapter use BF16, matching the published Qwen3 checkpoint. The notebook requires CUDA BF16 support.

In [ ]:
SEED = 42
MODEL_DTYPE = torch.bfloat16

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 8
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQUENCE_LENGTH = 512

GENERATION_MAX_NEW_TOKENS = 128
TOKEN_F1_ACCURACY_THRESHOLD = 0.80

DEVICE = require_cuda_bfloat16_support()
set_seed(SEED)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA version: {torch.version.cuda}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')
print(f'CUDA device name: {torch.cuda.get_device_name(DEVICE)}')
print(f'Selected device: {DEVICE}')
print(f'Model dtype: {MODEL_DTYPE}')

In [ ]:
raw_datasets = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'eval': str(EVAL_FILE)})

for split_name in ('train', 'eval'):
    for record in raw_datasets[split_name]:
        roles = [message['role'] for message in record['messages']]
        if roles != ['system', 'user', 'assistant']:
            raise ValueError(f'Invalid roles in {split_name} record {record["id"]}: {roles}')
        all_content = ' '.join(message['content'] for message in record['messages']).lower()
        if 'luigi saetta' not in all_content or any(term in all_content for term in ('cv', 'candidate', 'document')):
            raise ValueError(f'Invalid parametric-memory wording in {split_name} record {record["id"]}.')

train_fact_ids = {metadata['fact_id'] for metadata in raw_datasets['train']['metadata']}
eval_fact_ids = {metadata['fact_id'] for metadata in raw_datasets['eval']['metadata']}
if not eval_fact_ids.issubset(train_fact_ids):
    raise ValueError(f'Evaluation facts missing from training: {sorted(eval_fact_ids - train_fact_ids)}')

train_questions = {record['messages'][1]['content'].strip() for record in raw_datasets['train']}
eval_questions = {record['messages'][1]['content'].strip() for record in raw_datasets['eval']}
if train_questions & eval_questions:
    raise ValueError('Train and evaluation splits share exact user-question strings.')

sft_datasets = raw_datasets.map(to_prompt_completion, remove_columns=raw_datasets['train'].column_names)
print(f'Train records: {len(sft_datasets["train"])}')
print(f'Evaluation records: {len(sft_datasets["eval"])}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIRECTORY,
    dtype=MODEL_DTYPE,
    local_files_only=True,
).to(DEVICE)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)

training_arguments = SFTConfig(
    output_dir=str(TRAINING_OUTPUT_DIRECTORY),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_length=MAX_SEQUENCE_LENGTH,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=5,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    completion_only_loss=True,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    dataloader_pin_memory=True,
    optim='adamw_torch',
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=sft_datasets['train'],
    eval_dataset=sft_datasets['eval'],
    processing_class=tokenizer,
)

floating_parameter_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.is_floating_point()}
trainable_parameter_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
if floating_parameter_dtypes != {MODEL_DTYPE}:
    raise RuntimeError(f'Expected all floating-point model parameters to use {MODEL_DTYPE}, got {floating_parameter_dtypes}.')
if trainable_parameter_dtypes != {MODEL_DTYPE}:
    raise RuntimeError(f'Expected all trainable LoRA parameters to use {MODEL_DTYPE}, got {trainable_parameter_dtypes}.')

trainer.model.print_trainable_parameters()
print(f'Model parameter dtype: {next(trainer.model.parameters()).dtype}')
print(f'Trainable LoRA parameter dtypes: {trainable_parameter_dtypes}')

## Train and evaluate loss at every epoch

`eval_strategy='epoch'` records validation loss after every completed epoch and restores the adapter with the lowest validation loss.

In [ ]:
train_result = trainer.train()

trainer.save_model(ADAPTER_OUTPUT_DIRECTORY)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIRECTORY)

epoch_evaluations = [
    {key: value for key, value in entry.items() if key in {'epoch', 'eval_loss', 'eval_runtime', 'eval_samples_per_second'}}
    for entry in trainer.state.log_history
    if 'eval_loss' in entry
]
print(f'Training loss: {train_result.training_loss:.4f}')
print('Evaluation loss by epoch:')
for evaluation in epoch_evaluations:
    print(evaluation)
print(f'Adapter saved to: {ADAPTER_OUTPUT_DIRECTORY}')

In [ ]:
training_logs = [entry for entry in trainer.state.log_history if 'loss' in entry and 'epoch' in entry]
validation_logs = [entry for entry in trainer.state.log_history if 'eval_loss' in entry and 'epoch' in entry]
if not training_logs or not validation_logs:
    raise RuntimeError('Training or validation loss logs are unavailable.')

fig, axis = plt.subplots(figsize=(10, 6))
axis.plot([entry['epoch'] for entry in training_logs], [entry['loss'] for entry in training_logs], color='#2563EB', linewidth=2, marker='o', markersize=4, label='Training loss')
axis.plot([entry['epoch'] for entry in validation_logs], [entry['eval_loss'] for entry in validation_logs], color='#DC2626', linewidth=2, marker='s', markersize=6, label='Validation loss')
axis.set_title('Training and Validation Loss by Epoch', fontsize=14, fontweight='bold')
axis.set_xlabel('Epoch')
axis.set_ylabel('Loss')
axis.grid(True, linestyle='--', linewidth=0.7, alpha=0.55)
axis.legend(frameon=True)
axis.set_xticks([entry['epoch'] for entry in validation_logs])
plt.tight_layout()
plt.show()

## Evaluate held-out response accuracy

This deterministic-generation check complements validation loss with exact-match and token-F1 metrics on unseen question phrasings.

In [ ]:
def normalise_for_scoring(text: str) -> list[str]:
    """Return lowercase word tokens for a lightweight factual-overlap metric."""
    return re.findall(r"\w+", text.casefold())


def token_f1(prediction: str, reference: str) -> float:
    """Calculate bag-of-token F1 between a generated and approved answer."""
    prediction_counts = Counter(normalise_for_scoring(prediction))
    reference_counts = Counter(normalise_for_scoring(reference))
    if not prediction_counts or not reference_counts:
        return 0.0
    overlap = sum((prediction_counts & reference_counts).values())
    precision = overlap / sum(prediction_counts.values())
    recall = overlap / sum(reference_counts.values())
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def generate_answer(messages: list[dict[str, str]]) -> str:
    """Generate a deterministic answer from system and user messages only."""
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    model_inputs = {name: tensor.to(DEVICE) for name, tensor in tokenizer(prompt_text, return_tensors='pt').items()}
    with torch.inference_mode():
        output_ids = trainer.model.generate(**model_inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output_ids[0, model_inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()


evaluation_results = []
trainer.model.eval()
for record in raw_datasets['eval']:
    reference = record['messages'][-1]['content']
    prediction = generate_answer(record['messages'])
    score = token_f1(prediction, reference)
    evaluation_results.append({'id': record['id'], 'prediction': prediction, 'reference': reference, 'exact_match': prediction.casefold().strip() == reference.casefold().strip(), 'token_f1': score, 'correct_at_threshold': score >= TOKEN_F1_ACCURACY_THRESHOLD})

exact_match_rate = sum(result['exact_match'] for result in evaluation_results) / len(evaluation_results)
mean_token_f1 = sum(result['token_f1'] for result in evaluation_results) / len(evaluation_results)
threshold_accuracy = sum(result['correct_at_threshold'] for result in evaluation_results) / len(evaluation_results)
print(f'Exact-match rate: {exact_match_rate:.1%}')
print(f'Mean token F1: {mean_token_f1:.3f}')
print(f'Accuracy at token-F1 >= {TOKEN_F1_ACCURACY_THRESHOLD:.2f}: {threshold_accuracy:.1%}')